# Análise Exploratória de Dados (EDA): Mobilidade Urbana no Município do Rio de Janeiro (1995–2025)

**Disciplina:** Linguagem de Programação: Análise e Visualização de Dados com Python (6º Período — Sistemas de Informação)  
**Fonte Oficial:** Data.Rio — Tabela 728 (*Movimento médio diário de passageiros, segundo os transportes rodoviário, ferroviário, hidroviário e aeroviário no Município do Rio de Janeiro entre 1995-2025*, em milhares de passageiros/dia).

### Questões de Pesquisa (RQs)
- **RQ1 (Transição de *Modal Share*):** Como evoluiu a participação relativa dos modais ao longo de 31 anos (1995–2025), em especial o peso dos ônibus frente aos sistemas ferroviários?
- **RQ2 (Choque da Pandemia e Recuperação):** Qual foi a magnitude da queda na demanda diária em 2020–2021 em relação a 2019 (`2019 = 100`) e qual o grau de recuperação até 2025?
- **RQ3 (Ciclo de Megaeventos):** Como se comportou a demanda agregada e modal durante o ciclo de preparação e realização dos megaeventos no Rio de Janeiro (2007–2016)?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

PROCESSED_DIR = Path('data/processed')
PARQUET_PATH = PROCESSED_DIR / 'passageiros_rio_1995_2025.parquet'
CSV_GZ_PATH = PROCESSED_DIR / 'passageiros_rio_1995_2025.csv.gz'
CSV_PATH = PROCESSED_DIR / 'passageiros_rio_1995_2025.csv'
RAW_PATH = Path('data/raw/728.xls')

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
elif CSV_GZ_PATH.exists():
    df = pd.read_csv(CSV_GZ_PATH, compression='gzip')
elif CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
else:
    from src.etl import run_pipeline
    df = run_pipeline()

print('Dimensões (linhas, colunas):', df.shape)
print('Intervalo de anos:', int(df['ano'].min()), 'a', int(df['ano'].max()))
df.head()


## 1. Qualidade dos Dados, Valores Ausentes e Estatísticas Descritivas

Verificamos tipos primitivos, presença de valores ausentes (`NaN`) — com atenção especial a `bonde_mil` entre **2012 e 2014**, período de paralisação após o acidente de agosto de 2011 até a retomada em julho de 2015 — e estatísticas resumidas.


In [ ]:
print('Valores ausentes por coluna:')
na_counts = df.isna().sum()
print(na_counts[na_counts > 0])
print('\nAnos com bonde_mil ausente (NaN):', df.loc[df['bonde_mil'].isna(), 'ano'].tolist())

desc_cols = [
    'onibus_mil', 'metro_mil', 'trem_mil', 'bonde_mil',
    'hidroviario_mil', 'aeroviario_mil', 'ferroviario_total_mil', 'total_geral_mil'
]
stats = df[desc_cols].agg(['count', 'mean', 'median', 'min', 'max', 'std']).T
stats.round(2)


## 2. RQ1 — Transição de *Modal Share* (1995–2025)

Analisamos a participação percentual (`share_*_pct`) de cada modal no total diário de passageiros e a evolução comparativa entre o modal rodoviário (`onibus_mil`) e os modais ferroviários (`ferroviario_total_mil`).


In [ ]:
marcos_anos = [1995, 2005, 2015, 2019, 2020, 2021, 2025]
share_cols = [
    'ano', 'total_geral_mil', 'share_onibus_pct', 'share_ferroviario_pct',
    'share_metro_pct', 'share_trem_pct', 'share_hidroviario_pct', 'share_aeroviario_pct'
]
df.loc[df['ano'].isin(marcos_anos), share_cols].round(2)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

stack_cols = [
    'share_onibus_pct', 'share_metro_pct', 'share_trem_pct',
    'share_hidroviario_pct', 'share_aeroviario_pct', 'share_bonde_pct'
]
labels_stack = ['Ônibus', 'Metrô', 'Trem', 'Hidroviário', 'Aeroviário', 'Bonde']
ax1.stackplot(df['ano'], [df[c].fillna(0) for c in stack_cols], labels=labels_stack, alpha=0.88)
ax1.set_title('Composição Percentual do Modal Share (1995–2025)')
ax1.set_xlabel('Ano')
ax1.set_ylabel('Participação no Total Diário (%)')
ax1.set_ylim(0, 100)
ax1.legend(loc='lower left', ncol=2, fontsize=8, frameon=True)

ax2.plot(df['ano'], df['share_onibus_pct'], marker='o', markersize=4, label='Ônibus (%)')
ax2.plot(df['ano'], df['share_ferroviario_pct'], marker='s', markersize=4, label='Ferroviário Total (%)')
ax2.set_title('Ônibus vs. Sistemas Ferroviários (% do Total)')
ax2.set_xlabel('Ano')
ax2.set_ylabel('Participação (%)')
ax2.legend(frameon=True)

plt.tight_layout()
plt.show()


## 3. RQ2 — Choque da Pandemia (2020–2021) e Recuperação até 2025 (`2019 = 100`)

Comparamos o nível de demanda em cada modal e no total geral usando `2019 = 100` como referência pré-pandemia.


In [ ]:
idx_cols = [
    'ano', 'idx_2019_total_geral', 'idx_2019_onibus', 'idx_2019_metro',
    'idx_2019_trem', 'idx_2019_hidroviario', 'idx_2019_aeroviario', 'idx_2019_bonde'
]
df.loc[df['ano'].between(2018, 2025), idx_cols].round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.axhline(100, color='black', linestyle='--', linewidth=1, label='Referência 2019 (= 100)')
ax.axvspan(2020, 2021, color='salmon', alpha=0.18, label='Choque Pandêmico (2020–2021)')

plot_idx = [
    ('idx_2019_total_geral', 'Total Geral', '-o'),
    ('idx_2019_onibus', 'Ônibus', '-s'),
    ('idx_2019_metro', 'Metrô', '-^'),
    ('idx_2019_trem', 'Trem', '-d'),
    ('idx_2019_aeroviario', 'Aeroviário', '-P'),
]
for col, lbl, fmt in plot_idx:
    ax.plot(df['ano'], df[col], fmt, markersize=4.5, label=lbl)

ax.set_xlim(2015, 2025)
ax.set_title('Índice de Demanda Pré/Pós-Pandemia (Base 2019 = 100)')
ax.set_xlabel('Ano')
ax.set_ylabel('Índice (2019 = 100)')
ax.legend(ncol=2, fontsize=8.5, frameon=True)
plt.tight_layout()
plt.show()


## 4. RQ3 — Ciclo de Megaeventos (2007–2016) e Síntese dos Achados

Observamos a trajetória da demanda agregada (`total_geral_mil`) e dos principais modais entre o Pan-Americano de 2007, a Copa do Mundo de 2014 e os Jogos Olímpicos/Paralímpicos de 2016.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df['ano'], df['total_geral_mil'], color='#1f4e78', linewidth=2.2, marker='o', markersize=4, label='Total Geral (mil/dia)')
ax.plot(df['ano'], df['onibus_mil'], color='#2e7d32', linewidth=1.6, linestyle='--', marker='s', markersize=3.5, label='Ônibus (mil/dia)')
ax.plot(df['ano'], df['ferroviario_total_mil'], color='#c62828', linewidth=1.6, linestyle='-.', marker='^', markersize=3.5, label='Ferroviário Total (mil/dia)')

eventos = [(2007, 'Pan 2007'), (2014, 'Copa 2014'), (2016, 'Rio 2016')]
for ano_ev, rotulo in eventos:
    y_val = df.loc[df['ano'] == ano_ev, 'total_geral_mil'].values[0]
    ax.axvline(ano_ev, color='gray', linestyle=':', linewidth=1)
    ax.annotate(rotulo, xy=(ano_ev, y_val), xytext=(ano_ev - 1.2, y_val + 250),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8), fontsize=8.5)

ax.set_title('Série Histórica de Passageiros/Dia e Marcos de Megaeventos (1995–2025)')
ax.set_xlabel('Ano')
ax.set_ylabel('Passageiros por dia (x1.000)')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

pico_ano = int(df.loc[df['total_geral_mil'].idxmax(), 'ano'])
pico_val = df['total_geral_mil'].max()
vale_ano = int(df.loc[df['total_geral_mil'].idxmin(), 'ano'])
vale_val = df['total_geral_mil'].min()
print(f'Pico da série: {pico_val:.2f} mil/dia em {pico_ano} | Vale da série: {vale_val:.2f} mil/dia em {vale_ano}')
